<a href="https://colab.research.google.com/github/Jn-Wng/TFG-JinWang/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Descripción del Dataset y su Estructura
El dataset utilizado en este trabajo es el Chest X-Ray Pneumonia, disponible públicamente en la plataforma Kaggle (https://teams.microsoft.com/l/message/48:notes/1777446558878?context=%7B%22contextType%22%3A%22chat%22%2C%22oid%22%3A%228%3Aorgid%3Ab40e80d8-786b-43ad-af54-629fdf5d7de1%22%7D). Fue compilado por investigadores del Guangzhou Women and Children's Medical Center y contiene un total de 5.863 imágenes de radiografías de tórax en formato JPEG, distribuidas en dos clases: NORMAL y PNEUMONIA.

La estructura del dataset sigue una organización estándar orientada al entrenamiento de modelos supervisados:

chest_xray/

├── train/
│   ├── NORMAL/      (1.341 imágenes)
│    PNEUMONIA/   (3.875 imágenes)

├── val/
│   ├── NORMAL/      (8 imágenes)
│    PNEUMONIA/   (8 imágenes)

└── test/
    ├── NORMAL/      (234 imágenes)
    |     PNEUMONIA/   (390 imágenes)

Es importante señalar dos aspectos críticos de este dataset. En primer lugar, existe un notable desbalanceo de clases en el conjunto de entrenamiento: la clase PNEUMONIA triplica en número a la clase NORMAL, lo que puede inducir al modelo a sesgar sus predicciones hacia la clase mayoritaria. Esta problemática deberá abordarse mediante técnicas de ponderación de clases o sobremuestreo durante el entrenamiento. En segundo lugar, el conjunto de validación oficial contiene únicamente 16 imágenes —8 por clase—, una cantidad insuficiente para obtener métricas de validación robustas. Por tanto, en la implementación práctica se optará por redefinir los conjuntos: se partirá el conjunto de entrenamiento original en un 80% para entrenamiento efectivo y un 20% para validación, empleando el conjunto de test oficial exclusivamente para la evaluación final del modelo.

# Importación de Librerías

In [ ]:
# !pip install tensorflow keras scikit-learn matplotlib seaborn lime shap


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=8e9a02dcaa11a89164da23bb1ee8a292f61edd95d4ea21746f21867c3efb7fd2
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
# ── Utilidades del sistema ─────────────────────────────────────────────────
import os
import random
import warnings
warnings.filterwarnings('ignore')

# ── Manipulación de datos y visualización ──────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# ── TensorFlow / Keras ─────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)

# ── Scikit-learn (métricas y utilidades) ───────────────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight

# ── Explicabilidad ─────────────────────────────────────────────────────────
import lime
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
import shap

# ── Reproducibilidad ───────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [ ]:
print(f"TensorFlow versión : {tf.__version__}")
print(f"Keras versión      : {keras.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✓ GPU disponible   : {len(gpus)} dispositivo(s)")
    for gpu in gpus:
        print(f"  └─ {gpu.name}")
    # Habilitar crecimiento de memoria dinámico (evita reservar toda la VRAM)
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("\n No se detectó GPU. El entrenamiento se realizará en CPU (lento).")
    print("  → Considera usar Google Colab con runtime GPU.")

TensorFlow versión : 2.20.0
Keras versión      : 3.13.2

✓ GPU disponible   : 1 dispositivo(s)
  └─ /physical_device:GPU:0


Define en un único lugar todos los hiperparámetros: rutas, tamaño de imagen, batch size, épocas y tasas de aprendizaje de cada fase. Cambiar un valor aquí se propaga a todo el cuaderno.

In [ ]:
# ── Rutas del dataset ──────────────────────────────────────────────────────
BASE_DIR   = 'chest_xray'       # Raíz del dataset de Kaggle
TRAIN_DIR  = os.path.join(BASE_DIR, 'train')
VAL_DIR    = os.path.join(BASE_DIR, 'val')
TEST_DIR   = os.path.join(BASE_DIR, 'test')

# ── Parámetros de imagen ───────────────────────────────────────────────────
IMG_SIZE    = (224, 224)   # Tamaño estándar de ResNet50
CHANNELS    = 3            # RGB (aunque las RX son grises, se replica en 3 canales)
INPUT_SHAPE = IMG_SIZE + (CHANNELS,)

# ── Parámetros de entrenamiento ────────────────────────────────────────────
BATCH_SIZE     = 32
EPOCHS_PHASE1  = 10    # Feature extraction (capas congeladas)
EPOCHS_PHASE2  = 25    # Fine-tuning (capas descongeladas)
LR_PHASE1      = 1e-3
LR_PHASE2      = 1e-5

# ── Clases ─────────────────────────────────────────────────────────────────
CLASSES     = ['NORMAL', 'PNEUMONIA']
NUM_CLASSES = 1   # Salida binaria con sigmoide

Configuración del experimento:
  Tamaño de imagen  : (224, 224)
  Batch size        : 32
  Épocas fase 1     : 10
  Épocas fase 2     : 25
  LR fase 1         : 0.001
  LR fase 2         : 1e-05


# Exploración del Dataset

## Distribución del dataset por split y clase

In [ ]:
def contar_imagenes(directorio):
    """Devuelve un dict {clase: n_imágenes} para un directorio dado."""
    conteo = {}
    for clase in sorted(os.listdir(directorio)):
        ruta_clase = os.path.join(directorio, clase)
        if os.path.isdir(ruta_clase):
            n = len([
                f for f in os.listdir(ruta_clase)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
            conteo[clase] = n
    return conteo

splits = {
    'Train' : contar_imagenes(TRAIN_DIR),
    'Val'   : contar_imagenes(VAL_DIR),
    'Test'  : contar_imagenes(TEST_DIR),
}

print(f"{'Split':<10} {'NORMAL':>10} {'PNEUMONIA':>12} {'Total':>8}")
print("-" * 42)
for split, conteo in splits.items():
    n   = conteo.get('NORMAL', 0)
    p   = conteo.get('PNEUMONIA', 0)
    tot = n + p
    ratio = p / n if n > 0 else float('inf')
    print(f"{split:<10} {n:>10,} {p:>12,} {tot:>8,}  (ratio P/N: {ratio:.2f})")

total_train = sum(splits['Train'].values())
print(f"\n→ Desbalanceo en train: la clase PNEUMONIA es ~{splits['Train']['PNEUMONIA']/splits['Train']['NORMAL']:.1f}x más frecuente que NORMAL.")
print(f"→ Val oficial: solo {sum(splits['Val'].values())} imágenes. Se reemplazará por split interno del train.")

FileNotFoundError: [Errno 2] No such file or directory: 'chest_xray/train'